In [1]:
try:
    get_ipython().run_line_magic('load_ext' , 'autoreload')
    get_ipython().run_line_magic('autoreload', 2)
except:
    pass

import scipy.io as sio
import datetime
import h5py
import glob
import fnmatch
import os
import sys


import logging
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.ndimage import gaussian_filter
from tifffile.tifffile import imwrite

from extract_models import extract_2d
import caiman as cm
from caiman.utils.visualization import plot_contours, nb_plot_contour, nb_view_patches3d
import caiman.source_extraction.cnmf as cnmf
from caiman.paths import caiman_datadir
from caiman.source_extraction.cnmf import params as params

import bokeh.plotting as bpl
bpl.output_notebook()

logging.basicConfig(format=
                          "%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s] [%(process)d] %(message)s",
                    # filename="/tmp/caiman.log",
                    #level=logging.INFO,
                    )


def tracefunc(frame, event, arg, indent=[0]): # can get line number with frame.f_lineno
  strmtch = '/Users/wienecke/mambaforge/envs/caiman/lib/python3.10/site-packages/caiman' 
  if event == "call":
      if strmtch in frame.f_code.co_filename:
        indent[0] += 2
        print("-" * indent[0] + "> call function", frame.f_code.co_filename)
        print("-" * indent[0] + "> call function", frame.f_code.co_name)
      
  elif event == "return":
      if strmtch in frame.f_code.co_filename:
        print("<" + "-" * indent[0], "exit function", frame.f_code.co_name)
        print("<" + "-" * indent[0], "exit function", frame.f_code.co_filename)
        indent[0] -= 2
  return tracefunc

2023-07-28 20:27:10.178394: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading BokehJS ...

In [4]:
a=np.arange(12).reshape((3,4))
np.array_equal(a.shape, a.shape)


True

In [ ]:
fname_mmap_reg = ['/Users/wienecke/Documents/ambrose/leprechaunMat/20230627-2_D05_syt7f_018_syt7f/20230627_2_3047_15_140_256_TMPcaimanregTMPtrial_002_00001_rig__d1_256_d2_140_d3_15_order_F_frames_3047.mmap']
Yr, dims_original, T = cm.load_memmap(fname_mmap_reg[0]) #np.allclose(images, images2, rtol=1e-05, atol=1e-04, equal_nan=False)
images = np.reshape(Yr.T, [T] + list(dims_original), order='F')
images=np.moveaxis(images,1,0).copy()
#images = np.reshape(images.T, [T] + list(dims_original), order='C') 
Yr = np.transpose(np.reshape(images, (T, -1), order='F'))

print(np.isfortran(Yr))


In [ ]:
def configs(index = None):

    only_init = False

    strides_mc = (24, 24, 6)
    overlaps_mc = (12, 12, 3)
    max_shifts_mc = (4, 4, 2)
    max_deviation_rigid = 3
    pw_rigid = False
    is3D_mc = True
    nonneg_movie = True
    min_mov = 0
    shifts_opencv = True 
    indices_mc = (slice(None), slice(None), slice(None)) #if is3d is true for motion correction, will overwrite with nones and will lose indices_ex


    sly = slice(0, 51, 1)
    slx = slice(40, 181, 1) 
    slz = slice(0, 7, 1)
    indices_ex = [slx, sly, slz]
    #indices_ex = [slice(None), slice(None), slice(None)]

    p = 0                   # order of the autoregressive system - 0 from carl's code
    merge_thresh = 0.4
    gSig = [3, 3, 3]  # gSig = [3,3]            # radius (half-size) of average neurons (in pixels)
    nb = 1

    do_patches = False      # flag for processing in patches or not - turn on or off - Not used in Matlab
    if do_patches:          # PROCESS IN PATCHES AND THEN COMBINE
        rf = 60             # half size of each patch
        stride_cnmf = 40          # overlap between patches
        p_patch = p
        nb_patch = nb
        k = 3              # number of components in each patch
        indices_ex = [slice(None), slice(None), slice(None)]
    else:                   # PROCESS THE WHOLE FOV AT ONCE
        rf = None           # setting these parameters to None
        stride_cnmf = None       # will run CNMF on the whole FOV
        p_patch = p
        nb_patch = nb
        k = 30              # number of neurons expected (in the whole FOV) - 40 from Carl's Code, seems to be too many

    ###
    fr = 5.08 #0.6193  #9.8465 frame period so 1000 / (9.8465 *(113+51)) # approximate frame rate of data - CONFIRMED FPS
    decay_time = .4         # length of transient - CONFIRMED APPROPRIATE FOR OUR INDICATOR GCaMP6f
    dxy = [1.33155792277, 1.33155792277, 0.16666666666666666] #for .751 um pixels # pixels per micron 

    tsub = 1                # temporal downsampling
    ssub = 1               # spatial downsampling

    method_init = 'greedy_roi' #'graph_nmf' #'greedy_roi' #python Caiman defaults to greedy_roi, carl's code uses sparse_nmf, but sparse_nmf runs MUCH slower
    sigma_smooth_snmf = (0.5, 0.5, 0.5)
    perc_baseline_snmf = 50

    se = np.ones((3,)*len(gSig), dtype=np.uint8)  #se = np.ones((3,3,1), dtype=np.uint8)
    update_background_components = True   #use this??

    fudge_factor = 0.96        # (default is 0.96; Carl's value = 1) -- bias correction factor for discrete time constants
    ITER = 5                # (default is 2; Carl's value=5) -- block coordinate descent iterations
    bas_nonneg = False #True

    rolling_sum = True

    min_SNR = 0  #2    # accept components with that peak-SNR or higher (if above this, acept)
    SNR_lowest = 0         # minimum SNR for accepted components (if below this, reject)
    rval_thr = 0  # 0.85  # space correlation threshold (if above this, accept)
    rval_lowest = 0  # 0.6  # space correlation threshold (if above this, accept)
    use_cnn = False      # use the CNN classifier affects if 2 below params are used
    min_cnn_thr = 0 #0.99 # if cnn classifier predicts below this value, reject
    cnn_lowest = 0  #0.1  # neurons with cnn probability lower than this value are rejected

    fnadd = '_' + str(gSig[0]) + '_' + str(nb) + '_' + str(merge_thresh) + '_' + str(k) + '_'
    print("index is " + str(index) + " with filename string " + fnadd)
    
    gSiz = [2*gstmp + 1 for gstmp in gSig] #this should be after map2params to make sure it registers the gSig change
    
    opts_dict = {'strides': strides_mc,    # start a new patch for pw-rigid motion correction every x pixels
                'overlaps': overlaps_mc,   # overlap between pathes (size of patch strides+overlaps)
                'max_shifts': max_shifts_mc,   # maximum allowed rigid shifts (in pixels)
                'max_deviation_rigid': max_deviation_rigid,  # maximum shifts deviation allowed for patch with respect to rigid shifts
                'pw_rigid': pw_rigid,         # flag for performing non-rigid motion correction
                'is3D': is3D_mc,
                'nonneg_movie':nonneg_movie,
                'min_mov':min_mov,
                'shifts_opencv':shifts_opencv, 
                'fr': fr,
                'p': p,
                'nb': nb,
                'merge_thr': merge_thresh,
                'rf': rf,
                'indices': indices_mc,  #for some reason indices_mc is causing error, maybe needs list for mc and tuple for extraction?
                'K': k, 
                'gSig': gSig,
                'gSiz': gSiz,
                'stride': stride_cnmf,
                'method_init': method_init,
                'sigma_smooth_snmf': sigma_smooth_snmf,
                'perc_baseline_snmf': perc_baseline_snmf,
                #'dims': dims,
                'dxy': dxy,
                'decay_time': decay_time,
                'bas_nonneg': bas_nonneg,
                'p_patch': p_patch,
                'nb_patch': nb_patch,
                'se': se, 
                'rolling_sum': rolling_sum,
                'only_init': only_init,
                'ssub': ssub,
                'tsub': tsub,
                'SNR_lowest': SNR_lowest, 
                'min_SNR': min_SNR,
                'rval_thr': rval_thr,
                'rval_lowest': rval_lowest,
                'use_cnn': use_cnn,
                'min_cnn_thr': min_cnn_thr,
                'cnn_lowest': cnn_lowest, 
                'ITER': ITER, 
                'fudge_factor': fudge_factor, 
                'cnn_lowest': cnn_lowest, 
                'update_background_components': update_background_components}

    return opts_dict, indices_ex, fnadd


In [ ]:

fname_mmap_reg = ['/Users/wienecke/Documents/ambrose/leprechaunMat/20230627-2_D05_syt7f_018_syt7f/20230627_2_3047_15_140_256_TMPcaimanregTMPtrial_002_00001_rig__d1_256_d2_140_d3_15_order_F_frames_3047.mmap']

if 'dview' in locals():
    cm.stop_server(dview=dview)
c, dview, n_processes = cm.cluster.setup_cluster(
    backend='local', n_processes=None, single_thread=False)

opts_dict, indices_ex, fnadd = configs(index = None)

Yr, dims, T = cm.load_memmap(fname_mmap_reg[0]) #np.allclose(images, images2, rtol=1e-05, atol=1e-04, equal_nan=False)
images = np.reshape(Yr.T, [T] + list(dims), order='F') 
min_mov = np.min(images)
print("MIN AFTER MOTION CORRECTION")
print(min_mov)
Yr = None

opts = cnmf.params.CNMFParams(params_dict=opts_dict)
opts.change_params({'fnames': fname_mmap_reg, 'min_mov': min_mov}) #i don't understand why i have to pass fname_tmptif_reg to motioncorrect and set in params object but i do 


In [ ]:
do_planar_extraction = False
do_plots = True

if do_planar_extraction: #adjust images and some params for planar 
    
    sliceindz = np.arange(images.shape[-1])

    indices_ex = indices_ex[:-1] #change from 3d to 2d 

    dims_mask_space = (dims[1], dims[0])
    opts_dict_2d = { #change some params from 3d to 2d if doing planar extraction 
        'dxy': opts.data['dxy'][:-1],
        #'indices': indices_ex, 
        'sigma_smooth_snmf' : opts.init['sigma_smooth_snmf'][:-1],
        'gSig' : opts.init['gSig'][:-1],
        'gSiz' : opts.init['gSiz'][:-1],
        'se' : np.ones((3,)*len(opts.init['gSig'][:-1]), dtype=np.uint8)} 
    
    opts.change_params(opts_dict_2d)

else:
    
    sliceindz = [np.arange(images.shape[-1])] #all slices (not planar)
    dims = (dims[0], dims[1], dims[2]-1)
    dims_mask_space = (dims[1], dims[0], dims[2])


In [ ]:

if do_planar_extraction:
    images_sliced2 = images[:,:,:,sliceindz[4]]
else:
    images_sliced = images[:,:,:,1:]

cnm = cnmf.CNMF(n_processes, params=opts, dview=dview)
cnm = cnm.fit(images_sliced, indices = indices_ex)

if do_planar_extraction:

    if do_plots==1:
        Cn = cm.local_correlations(images_sliced.transpose(1,2,0))
        print(Cn.shape)
        Cn[np.isnan(Cn)] = 0
        print('you may need to change the data rate to generate nb_view_components: use jupyter notebook --NotebookApp.iopub_data_rate_limit=1.0e10 before opening jupyter notebook')                
        cnm.estimates.plot_contours(img=Cn)
        cnm.estimates.nb_view_components(img=Cn)
        #cnm.estimates.view_components(img=None, idx=cnm.estimates.idx_components) #img=None for mean projection

else:
    
    if do_plots==1:
        cnm.estimates.nb_view_components_3d(image_type='mean', dims=dims, axis=2)
        #cnm.estimates.nb_view_components_3d(image_type='corr', dims=dims, Yr=Yr, denoised_color='red', max_projection=True);



In [ ]:
cnm.estimates.evaluate_components(images_sliced, cnm.params, dview=dview)
print(('NUMGOOD ' + str(len(cnm.estimates.idx_components)) + ' NUMBAD ' + str(len(cnm.estimates.idx_components_bad))))
#cnm.estimates.select_components(use_object=True, save_discarded_components=True)

In [ ]:
do_refit = True

if do_refit:

    if 'dview' in locals():
        cm.stop_server(dview=dview)
    c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False)
    
    cnm2 = cnm.refit(images_sliced)
    cnm2.estimates.evaluate_components(images_sliced, cnm2.params, dview=dview)
    print(('REFIT: NUMGOOD ' + str(len(cnm2.estimates.idx_components)) + ' NUMBAD ' + str(len(cnm2.estimates.idx_components_bad))))
    cnm2.estimates.detrend_df_f(detrend_only=False, flag_auto=True, use_residuals=True) #use_residuals to use nondeconvolved traceds for dff computation (if p=0 use_residuals should not matter right?)
    #cnm2.estimates.select_components(use_object=True, save_discarded_components=True)

    if do_plots==1:
        if do_planar_extraction:
            cnm2.estimates.view_components(img=Cn) #img=Cn for mean proj
        else:
            cnm2.estimates.nb_view_components_3d(image_type='mean', dims=dims, axis=2)
            #cnm2.estimates.nb_view_components_3d(image_type='corr', dims=dims, Yr=Yr, denoised_color='red', max_projection=True);
        cnm2.estimates.play_movie(images_sliced, q_max=99.9, gain_res=2, magnification=2, bpx=True, include_bck=False, save_movie=True)


#denoised_movie = cm.movie(cnm2.estimates.A.dot(cnm2.estimates.C) + \
#            cnm2.estimates.b.dot(cnm2.estimates.f)).reshape(dims + (-1,), order='F').transpose([3, 0, 1, 2]) #%% reconstruct denoised movie 
